<a href="https://www.kaggle.com/code/tuannm3823/multi-class-food-recognition?scriptVersionId=322300196" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Multi-Class Food Recognition

This notebook builds a Food-101 image classifier and compares transfer learning
against deeper fine-tuning. The workflow is designed for Kaggle execution: data
is read from Kaggle input mounts, checkpoints and evaluation artifacts are
written to `/kaggle/working`, and the committed notebook remains lightweight
with outputs cleared.

## 1. Project Summary

Food-101 is a fine-grained visual categorization dataset containing 101 balanced
food classes. The classification problem is challenging because food images have
high presentation variance, inconsistent lighting, and strong visual overlap
between some categories.

The experiment has two stages:

| Stage | Purpose |
| --- | --- |
| Part A: Transfer learning | Freeze ImageNet-pretrained backbones and train a 3-layer classifier head for GoogLeNet, ResNet50, and MobileNetV3 Large. |
| Part B: Fine-tuning | Select the strongest Part A model and unfreeze deeper ResNet50 blocks to adapt visual features to Food-101. |

The target metric is top-1 validation accuracy, with per-class F1 used for error
analysis.


## 2. Runtime, Imports, And Configuration

All libraries are imported once in this section. Configuration values are kept in
`CFG` so the notebook can switch between full training and checkpoint-based
inference without editing downstream cells. For faster reruns, upload the
trained `.pth` files from `/kaggle/working/results` as a Kaggle dataset and set
`CFG.MODE = "inference"` with `CFG.ARTIFACT_DIR` pointing to that dataset.

In [ ]:
import random
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

print(f"PyTorch version: {torch.__version__}")

In [ ]:
@dataclass(frozen=True)
class CFG:
    """Central notebook configuration."""

    MODE: str = "train"  # Options: "train" or "inference"
    SEED: int = 42
    BATCH_SIZE: int = 32
    IMAGE_SIZE: tuple[int, int] = (224, 224)
    NUM_CLASSES: int = 101
    NUM_WORKERS: int = 2
    LEARNING_RATE: float = 1e-3
    FINE_TUNE_LEARNING_RATE: float = 1e-5
    TRANSFER_EPOCHS: int = 5
    FINE_TUNE_EPOCHS: int = 5
    TOP_K: int = 5
    HARD_CLASS_COUNT: int = 12
    ERROR_EXAMPLE_COUNT: int = 12
    LATENCY_WARMUP_RUNS: int = 10
    LATENCY_BENCHMARK_RUNS: int = 50
    SELECTED_ARCHITECTURE: str = "ResNet50"
    SELECTED_FINETUNE_EXPERIMENT: str = "Exp_2_Layer3_4"
    LOCAL_ROOT: Path = Path("/kaggle/working")
    RESULTS_DIR: Path = Path("/kaggle/working/results")
    FIGURES_DIR: Path = Path("/kaggle/working/results/figures")
    ARTIFACT_DIR: Path = Path("/kaggle/input/food101-baseline-artifacts")
    DATA_DIR: Path = Path("/kaggle/input/datasets/kmader/food41")


assert CFG.MODE in {"train", "inference"}, "CFG.MODE must be 'train' or 'inference'."
CFG.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CFG.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Execution device: GPU ({torch.cuda.get_device_name(0)})")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Execution device: Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Execution device: CPU")

print(f"Notebook mode: {CFG.MODE}")
print(f"Artifact directory: {CFG.ARTIFACT_DIR}")

## 3. Data Ingestion And Audit

The Food-101 Kaggle dataset is expected to contain an `images/` directory where
each class has its own folder. The manifest converts that folder structure into
a DataFrame with image paths and labels, which makes stratified splitting and
EDA straightforward.


In [ ]:
def create_data_manifest(image_dir: Path) -> pd.DataFrame:
    """Create an image-path and label manifest from class folders.

    Args:
        image_dir: Directory containing one subdirectory per class.

    Returns:
        DataFrame with `path` and `label` columns.
    """
    records: list[dict[str, str]] = []
    class_dirs = sorted(path for path in image_dir.iterdir() if path.is_dir())
    for class_dir in class_dirs:
        for image_path in sorted(class_dir.iterdir()):
            if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                records.append({"path": str(image_path), "label": class_dir.name})

    return pd.DataFrame.from_records(records)


DATA_DIR = CFG.DATA_DIR
IMAGE_DIR = DATA_DIR / "images"
if not IMAGE_DIR.exists():
    raise FileNotFoundError(
        "Food-101 images were not found at the configured Kaggle path: "
        f"{IMAGE_DIR}"
    )

df = create_data_manifest(IMAGE_DIR)

print(f"Food-101 root: {DATA_DIR}")
print(f"Image directory: {IMAGE_DIR}")
print(f"Total samples: {len(df):,}")
print(f"Total classes: {df['label'].nunique():,}")

In [ ]:
label_counts = df["label"].value_counts().sort_index()
class_summary = pd.DataFrame(
    {
        "class_name": label_counts.index,
        "image_count": label_counts.values,
    }
)

print("Class count summary:")
print(label_counts.describe().round(2).to_string())

if label_counts.nunique() == 1:
    print(
        "Food-101 is perfectly balanced: "
        f"{label_counts.iloc[0]:,} images per class across "
        f"{len(label_counts):,} classes."
    )
else:
    display(class_summary.sort_values("image_count").head(10))
    display(class_summary.sort_values("image_count", ascending=False).head(10))

In [ ]:
def sample_image_shapes(manifest: pd.DataFrame, sample_size: int = 500) -> pd.DataFrame:
    """Inspect image dimensions for a reproducible sample.

    Args:
        manifest: DataFrame with a `path` column.
        sample_size: Maximum number of images to inspect.

    Returns:
        DataFrame with sampled image widths and heights.
    """
    sample = manifest.sample(
        n=min(sample_size, len(manifest)),
        random_state=CFG.SEED,
    )
    shapes = []
    for image_path in sample["path"]:
        with Image.open(image_path) as image:
            shapes.append(image.size)

    return pd.DataFrame(shapes, columns=["width", "height"])


shape_df = sample_image_shapes(df)
print(shape_df.describe().loc[["min", "mean", "max"]].round(1).to_string())


## 4. Preprocessing, Splits, And Dataloaders

Training images receive light geometric augmentation. Validation, test, and
inference images use deterministic resizing and ImageNet normalization so model
comparisons remain consistent.


In [ ]:
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

TRAIN_TRANSFORMS = transforms.Compose(
    [
        transforms.Resize(CFG.IMAGE_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ]
)

VAL_TRANSFORMS = transforms.Compose(
    [
        transforms.Resize(CFG.IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ]
)

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=CFG.SEED,
    stratify=df["label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=CFG.SEED,
    stratify=temp_df["label"],
)

print(f"Train samples: {len(train_df):,}")
print(f"Validation samples: {len(val_df):,}")
print(f"Test samples: {len(test_df):,}")


In [ ]:
class FoodDataset(Dataset):
    """PyTorch dataset for Food-101 image classification."""

    def __init__(
        self,
        dataframe: pd.DataFrame,
        class_to_idx: dict[str, int],
        transform: transforms.Compose | None = None,
    ) -> None:
        """Initialize the dataset.

        Args:
            dataframe: Manifest with `path` and `label` columns.
            class_to_idx: Mapping from class name to numeric class index.
            transform: Optional torchvision transform pipeline.
        """
        self.df = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.classes = [name for name, _ in sorted(class_to_idx.items(), key=lambda item: item[1])]
        self.transform = transform

    def __len__(self) -> int:
        """Return the number of samples."""
        return len(self.df)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        """Load one transformed image and class index."""
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return image, self.class_to_idx[row["label"]]


class_names = sorted(df["label"].unique())
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}

train_dataset = FoodDataset(train_df, class_to_idx, transform=TRAIN_TRANSFORMS)
val_dataset = FoodDataset(val_df, class_to_idx, transform=VAL_TRANSFORMS)
test_dataset = FoodDataset(test_df, class_to_idx, transform=VAL_TRANSFORMS)

pin_memory = device.type == "cuda"
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=pin_memory,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=pin_memory,
)

batch_images, batch_labels = next(iter(train_loader))
print(f"Batch image shape: {tuple(batch_images.shape)}")
print(f"Batch label shape: {tuple(batch_labels.shape)}")


## 5. Model Construction

Each transfer-learning model uses ImageNet-pretrained features and the same
3-layer classifier head. This keeps the comparison focused on the backbone
rather than on differences in classifier capacity.


In [ ]:
def make_classifier_head(in_features: int, num_classes: int) -> nn.Sequential:
    """Build the project-standard 3-layer classifier head."""
    return nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, num_classes),
    )


def freeze_parameters(model: nn.Module) -> None:
    """Freeze all model parameters in place."""
    for parameter in model.parameters():
        parameter.requires_grad = False


def build_googlenet(num_classes: int, pretrained: bool = True) -> nn.Module:
    """Create GoogLeNet with a custom Food-101 classifier."""
    weights = models.GoogLeNet_Weights.DEFAULT if pretrained else None
    model = models.googlenet(weights=weights)
    freeze_parameters(model)
    model.fc = make_classifier_head(model.fc.in_features, num_classes)
    return model


def build_resnet50(num_classes: int, pretrained: bool = True) -> nn.Module:
    """Create ResNet50 with a custom Food-101 classifier."""
    weights = models.ResNet50_Weights.DEFAULT if pretrained else None
    model = models.resnet50(weights=weights)
    freeze_parameters(model)
    model.fc = make_classifier_head(model.fc.in_features, num_classes)
    return model


def build_mobilenet_v3(num_classes: int, pretrained: bool = True) -> nn.Module:
    """Create MobileNetV3 Large with a custom Food-101 classifier."""
    weights = models.MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
    model = models.mobilenet_v3_large(weights=weights)
    freeze_parameters(model)
    in_features = model.classifier[0].in_features
    model.classifier = make_classifier_head(in_features, num_classes)
    return model


def count_parameters(model: nn.Module) -> dict[str, int]:
    """Return total and trainable parameter counts."""
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
    return {"total": total, "trainable": trainable}


def checkpoint_path(name: str) -> Path:
    """Return the standard output checkpoint path."""
    return CFG.RESULTS_DIR / f"{name.lower()}.pth"


def resolve_checkpoint(name: str) -> Path:
    """Find a checkpoint in working results or attached Kaggle artifacts."""
    filename = f"{name.lower()}.pth"
    candidates = [
        CFG.RESULTS_DIR / filename,
        CFG.ARTIFACT_DIR / filename,
        CFG.ARTIFACT_DIR / "results" / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    if CFG.ARTIFACT_DIR.exists():
        matches = sorted(CFG.ARTIFACT_DIR.rglob(filename))
        if matches:
            return matches[0]

    raise FileNotFoundError(
        f"Checkpoint {filename} was not found in /kaggle/working/results "
        f"or attached artifact directory {CFG.ARTIFACT_DIR}."
    )

In [ ]:
if CFG.MODE == "train":
    transfer_models = {
        "GoogLeNet": build_googlenet(CFG.NUM_CLASSES).to(device),
        "ResNet50": build_resnet50(CFG.NUM_CLASSES).to(device),
        "MobileNetV3": build_mobilenet_v3(CFG.NUM_CLASSES).to(device),
    }

    parameter_rows = []
    for model_name, model in transfer_models.items():
        counts = count_parameters(model)
        parameter_rows.append(
            {
                "architecture": model_name,
                "total_parameters": counts["total"],
                "trainable_parameters": counts["trainable"],
            }
        )

    parameter_df = pd.DataFrame(parameter_rows)
    display(parameter_df)

    dummy_batch = torch.randn(1, 3, *CFG.IMAGE_SIZE).to(device)
    for model_name, model in transfer_models.items():
        output = model(dummy_batch)
        print(f"{model_name}: output shape {tuple(output.shape)}")
else:
    transfer_models = {}
    print("Skipping transfer model initialization in inference mode.")


## 6. Training And Evaluation Utilities

The training loop is shared by both transfer learning and fine-tuning. It tracks
loss and accuracy per epoch and saves the checkpoint with the best validation
accuracy.


In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    """Train one epoch and return average loss and accuracy."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        predicted = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / len(loader), 100.0 * correct / total


def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    """Evaluate a model and return average loss and accuracy."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            predicted = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / len(loader), 100.0 * correct / total


def history_to_frame(history: dict[str, list[float]], run_name: str) -> pd.DataFrame:
    """Convert training history to a tidy DataFrame."""
    history_df = pd.DataFrame(history)
    history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
    history_df.insert(0, "run_name", run_name)
    return history_df


def fit_model(
    model_name: str,
    model: nn.Module,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int,
    save_path: Path,
) -> dict[str, list[float]]:
    """Train a model and save the best validation checkpoint."""
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = -np.inf
    progress = tqdm(range(epochs), desc=model_name)

    for epoch in progress:
        start_time = time.time()
        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
        )
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        progress.set_postfix(
            {
                "train_acc": f"{train_acc:.1f}%",
                "val_acc": f"{val_acc:.1f}%",
                "val_loss": f"{val_loss:.3f}",
            }
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            elapsed = time.time() - start_time
            tqdm.write(
                f"Epoch {epoch + 1}: {model_name} improved to "
                f"{val_acc:.2f}% in {elapsed:.1f}s. Saved {save_path}."
            )

    history_to_frame(history, model_name).to_csv(
        CFG.RESULTS_DIR / f"history_{model_name.lower()}.csv",
        index=False,
    )
    return history

## 7. Part A: Transfer Learning Comparison

In training mode, this section trains the frozen-backbone baselines and records
validation performance for model selection. In inference mode, the section is
skipped because it is only needed to produce checkpoints.


In [ ]:
criterion = nn.CrossEntropyLoss()

if CFG.MODE == "train":
    transfer_optimizers = {
        model_name: optim.Adam(
            (parameter for parameter in model.parameters() if parameter.requires_grad),
            lr=CFG.LEARNING_RATE,
        )
        for model_name, model in transfer_models.items()
    }

    transfer_history = {}
    for model_name, model in transfer_models.items():
        save_path = checkpoint_path(f"best_model_{model_name}")
        transfer_history[model_name] = fit_model(
            model_name=model_name,
            model=model,
            optimizer=transfer_optimizers[model_name],
            criterion=criterion,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=CFG.TRANSFER_EPOCHS,
            save_path=save_path,
        )
else:
    transfer_history = {}
    print("Skipping transfer learning comparison in inference mode.")


In [ ]:
def plot_transfer_history(history: dict[str, dict[str, list[float]]]) -> None:
    """Plot validation accuracy and validation loss for transfer models."""
    if not history:
        print("No transfer history available to plot.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for model_name, model_history in history.items():
        epochs = range(1, len(model_history["val_acc"]) + 1)
        best_acc = max(model_history["val_acc"])
        axes[0].plot(
            epochs,
            model_history["val_acc"],
            marker="o",
            label=f"{model_name} ({best_acc:.1f}%)",
        )
        axes[1].plot(
            epochs,
            model_history["val_loss"],
            marker="s",
            label=model_name,
        )

    axes[0].set_title("Part A Validation Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy (%)")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].set_title("Part A Validation Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()


plot_transfer_history(transfer_history)


In [ ]:
def collect_classification_outputs(
    model: nn.Module,
    loader: DataLoader,
    class_names: list[str],
    top_k: int = CFG.TOP_K,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Collect predictions and summary metrics for a dataloader."""
    model.eval()
    rows = []
    top_1_correct = 0
    top_k_correct = 0
    total = 0
    manifest = getattr(loader.dataset, "df", None)

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probabilities = F.softmax(outputs, dim=1)
            top_probabilities, top_indices = torch.topk(probabilities, top_k, dim=1)
            predictions = top_indices[:, 0].cpu()

            labels_cpu = labels.cpu()
            top_1_correct += predictions.eq(labels_cpu).sum().item()
            top_k_correct += top_indices.cpu().eq(labels_cpu.unsqueeze(1)).any(dim=1).sum().item()

            batch_size = labels_cpu.size(0)
            start_idx = total
            total += batch_size

            for row_idx, true_idx in enumerate(labels_cpu.tolist()):
                pred_idx = predictions[row_idx].item()
                manifest_row = manifest.iloc[start_idx + row_idx] if manifest is not None else None
                rows.append(
                    {
                        "path": None if manifest_row is None else manifest_row["path"],
                        "true_idx": true_idx,
                        "true_label": class_names[true_idx],
                        "pred_idx": pred_idx,
                        "pred_label": class_names[pred_idx],
                        "confidence": top_probabilities[row_idx, 0].item(),
                        "is_correct": pred_idx == true_idx,
                    }
                )

    metrics_df = pd.DataFrame(
        [
            {"metric": "top_1_accuracy", "value": 100.0 * top_1_correct / total},
            {"metric": f"top_{top_k}_accuracy", "value": 100.0 * top_k_correct / total},
        ]
    )
    return pd.DataFrame(rows), metrics_df


def classification_report_frame(prediction_df: pd.DataFrame) -> pd.DataFrame:
    """Return a per-class classification report from prediction rows."""
    report = classification_report(
        prediction_df["true_label"],
        prediction_df["pred_label"],
        labels=class_names,
        output_dict=True,
        zero_division=0,
    )
    return pd.DataFrame(report).transpose().loc[class_names].sort_values(
        "f1-score",
        ascending=False,
    )


def confusion_pair_frame(prediction_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize the most frequent wrong class substitutions."""
    errors = prediction_df[~prediction_df["is_correct"]]
    if errors.empty:
        return pd.DataFrame(columns=["true_label", "pred_label", "count", "mean_confidence"])

    return (
        errors.groupby(["true_label", "pred_label"], as_index=False)
        .agg(count=("path", "size"), mean_confidence=("confidence", "mean"))
        .sort_values(["count", "mean_confidence"], ascending=False)
        .reset_index(drop=True)
    )


if CFG.MODE == "train":
    selected_transfer_model = transfer_models[CFG.SELECTED_ARCHITECTURE]
    selected_checkpoint = resolve_checkpoint(f"best_model_{CFG.SELECTED_ARCHITECTURE}")
    selected_transfer_model.load_state_dict(
        torch.load(selected_checkpoint, map_location=device)
    )
    transfer_pred_df, transfer_metrics_df = collect_classification_outputs(
        selected_transfer_model,
        val_loader,
        class_names,
    )
    transfer_report_df = classification_report_frame(transfer_pred_df)

    transfer_pred_df.to_csv(CFG.RESULTS_DIR / "transfer_val_predictions.csv", index=False)
    transfer_metrics_df.to_csv(CFG.RESULTS_DIR / "transfer_val_metrics.csv", index=False)
    transfer_report_df.to_csv(CFG.RESULTS_DIR / "transfer_val_class_report.csv")

    display(transfer_metrics_df)
    print("Top 10 classes by F1 score:")
    display(transfer_report_df.head(10)[["precision", "recall", "f1-score"]])
    print("Bottom 10 classes by F1 score:")
    display(transfer_report_df.tail(10)[["precision", "recall", "f1-score"]])
else:
    transfer_pred_df = pd.DataFrame()
    transfer_metrics_df = pd.DataFrame()
    transfer_report_df = pd.DataFrame()
    print("Skipping transfer-learning error analysis in inference mode.")

## 8. Part B: ResNet50 Fine-Tuning

ResNet50 is the selected fine-tuning candidate because it provides the strongest
transfer-learning accuracy in the saved baseline run. Two unfreezing depths
are compared:

| Experiment | Trainable layers |
| --- | --- |
| `Exp_1_Layer4` | ResNet `layer4` and classifier head |
| `Exp_2_Layer3_4` | ResNet `layer3`, `layer4`, and classifier head |


In [ ]:
def set_resnet_trainable_layers(model: nn.Module, trainable_prefixes: tuple[str, ...]) -> None:
    """Freeze all parameters except the requested ResNet prefixes."""
    for name, parameter in model.named_parameters():
        parameter.requires_grad = name.startswith(trainable_prefixes)


def load_resnet_for_finetuning(
    checkpoint: Path,
    trainable_prefixes: tuple[str, ...],
) -> nn.Module:
    """Load the Part A ResNet50 checkpoint and configure trainable layers."""
    model = build_resnet50(CFG.NUM_CLASSES, pretrained=False)
    model.load_state_dict(torch.load(checkpoint, map_location=device))
    set_resnet_trainable_layers(model, trainable_prefixes)
    return model.to(device)


if CFG.MODE == "train":
    resnet_checkpoint = resolve_checkpoint("best_model_ResNet50")
    fine_tune_experiments = {
        "Exp_1_Layer4": load_resnet_for_finetuning(
            resnet_checkpoint,
            trainable_prefixes=("layer4", "fc"),
        ),
        "Exp_2_Layer3_4": load_resnet_for_finetuning(
            resnet_checkpoint,
            trainable_prefixes=("layer3", "layer4", "fc"),
        ),
    }

    fine_tune_rows = []
    for experiment_name, model in fine_tune_experiments.items():
        counts = count_parameters(model)
        fine_tune_rows.append(
            {
                "experiment": experiment_name,
                "trainable_parameters": counts["trainable"],
                "frozen_parameters": counts["total"] - counts["trainable"],
            }
        )
    display(pd.DataFrame(fine_tune_rows))
else:
    fine_tune_experiments = {}
    print("Skipping fine-tuning model setup in inference mode.")


In [ ]:
if CFG.MODE == "train":
    fine_tune_history = {}
    for experiment_name, model in fine_tune_experiments.items():
        optimizer = optim.Adam(
            (parameter for parameter in model.parameters() if parameter.requires_grad),
            lr=CFG.FINE_TUNE_LEARNING_RATE,
        )
        fine_tune_history[experiment_name] = fit_model(
            model_name=experiment_name,
            model=model,
            optimizer=optimizer,
            criterion=criterion,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=CFG.FINE_TUNE_EPOCHS,
            save_path=checkpoint_path(f"finetuned_{experiment_name}"),
        )
else:
    fine_tune_history = {}
    print("Skipping fine-tuning loop in inference mode.")


In [ ]:
def plot_fine_tune_history(
    fine_tune_history: dict[str, dict[str, list[float]]],
    transfer_history: dict[str, dict[str, list[float]]],
) -> None:
    """Plot fine-tuning validation accuracy against the transfer baseline."""
    if not fine_tune_history:
        print("No fine-tuning history available to plot.")
        return

    plt.figure(figsize=(12, 5))
    for experiment_name, history in fine_tune_history.items():
        epochs = range(1, len(history["val_acc"]) + 1)
        plt.plot(epochs, history["val_acc"], marker="o", label=experiment_name)

    if CFG.SELECTED_ARCHITECTURE in transfer_history:
        baseline = max(transfer_history[CFG.SELECTED_ARCHITECTURE]["val_acc"])
        plt.axhline(
            baseline,
            color="gray",
            linestyle="--",
            label=f"{CFG.SELECTED_ARCHITECTURE} transfer baseline ({baseline:.1f}%)",
        )

    plt.title("Part B Fine-Tuning Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_fine_tune_history(fine_tune_history, transfer_history)


## 9. Final Model Evaluation And Inference

The final model uses the selected fine-tuned ResNet50 checkpoint. This section
adds the evaluation layer for the personal project: validation and held-out test
metrics, top-k accuracy, per-class reports, hard-class confusion diagnostics,
qualitative error examples, and model efficiency measurements.

In [ ]:
def load_final_model(checkpoint: Path) -> nn.Module:
    """Load the selected fine-tuned ResNet50 checkpoint."""
    model = build_resnet50(CFG.NUM_CLASSES, pretrained=False)
    model.load_state_dict(torch.load(checkpoint, map_location=device))
    return model.to(device)


def plot_hard_class_confusion(
    prediction_df: pd.DataFrame,
    report_df: pd.DataFrame,
    split_name: str,
) -> None:
    """Plot a normalized confusion matrix for the lowest-F1 classes."""
    hard_classes = report_df.tail(CFG.HARD_CLASS_COUNT).index.tolist()
    filtered = prediction_df[prediction_df["true_label"].isin(hard_classes)]
    matrix = confusion_matrix(
        filtered["true_label"],
        filtered["pred_label"],
        labels=hard_classes,
        normalize="true",
    )

    plt.figure(figsize=(12, 10))
    sns.heatmap(
        matrix,
        xticklabels=hard_classes,
        yticklabels=hard_classes,
        cmap="viridis",
        vmin=0,
        vmax=1,
        square=True,
    )
    plt.title(f"{split_name.title()} Confusion Matrix: Hardest Classes")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    figure_path = CFG.FIGURES_DIR / f"{split_name}_hard_class_confusion.png"
    plt.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()


def export_split_evaluation(
    model: nn.Module,
    loader: DataLoader,
    split_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Evaluate one split and export predictions, metrics, and diagnostics."""
    prediction_df, metrics_df = collect_classification_outputs(
        model,
        loader,
        class_names,
    )
    report_df = classification_report_frame(prediction_df)
    confusion_pairs_df = confusion_pair_frame(prediction_df)

    prediction_df.to_csv(CFG.RESULTS_DIR / f"{split_name}_predictions.csv", index=False)
    metrics_df.to_csv(CFG.RESULTS_DIR / f"{split_name}_metrics.csv", index=False)
    report_df.to_csv(CFG.RESULTS_DIR / f"{split_name}_class_report.csv")
    confusion_pairs_df.to_csv(
        CFG.RESULTS_DIR / f"{split_name}_confusion_pairs.csv",
        index=False,
    )

    print(f"{split_name.title()} metrics")
    display(metrics_df)
    print(f"{split_name.title()} hardest classes")
    display(report_df.tail(10)[["precision", "recall", "f1-score", "support"]])
    print(f"{split_name.title()} top confusion pairs")
    display(confusion_pairs_df.head(10))
    return prediction_df, metrics_df, report_df, confusion_pairs_df


def model_efficiency_summary(model: nn.Module) -> pd.DataFrame:
    """Measure parameter count, memory footprint, and single-image latency."""
    model.eval()
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    buffer_count = sum(buffer.numel() for buffer in model.buffers())
    size_mb = 4 * (parameter_count + buffer_count) / 1024**2
    sample = torch.randn(1, 3, *CFG.IMAGE_SIZE).to(device)

    with torch.no_grad():
        for _ in range(CFG.LATENCY_WARMUP_RUNS):
            _ = model(sample)
        if device.type == "cuda":
            torch.cuda.synchronize()
        start_time = time.perf_counter()
        for _ in range(CFG.LATENCY_BENCHMARK_RUNS):
            _ = model(sample)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start_time

    return pd.DataFrame(
        [
            {
                "parameters": parameter_count,
                "model_size_mb": size_mb,
                "device": str(device),
                "latency_ms_per_image": 1000 * elapsed / CFG.LATENCY_BENCHMARK_RUNS,
            }
        ]
    )


try:
    final_checkpoint = resolve_checkpoint(f"finetuned_{CFG.SELECTED_FINETUNE_EXPERIMENT}")
except FileNotFoundError as error:
    final_checkpoint = None
    print(error)

if final_checkpoint is not None:
    final_model = load_final_model(final_checkpoint)
    val_pred_df, val_metrics_df, val_report_df, val_confusion_pairs_df = export_split_evaluation(
        final_model,
        val_loader,
        "val",
    )
    (
        test_pred_df,
        test_metrics_df,
        test_report_df,
        test_confusion_pairs_df,
    ) = export_split_evaluation(
        final_model,
        test_loader,
        "test",
    )
    plot_hard_class_confusion(test_pred_df, test_report_df, "test")

    efficiency_df = model_efficiency_summary(final_model)
    efficiency_df.to_csv(CFG.RESULTS_DIR / "final_model_efficiency.csv", index=False)
    display(efficiency_df)
else:
    final_model = None
    val_pred_df = pd.DataFrame()
    val_metrics_df = pd.DataFrame()
    val_report_df = pd.DataFrame()
    val_confusion_pairs_df = pd.DataFrame()
    test_pred_df = pd.DataFrame()
    test_metrics_df = pd.DataFrame()
    test_report_df = pd.DataFrame()
    test_confusion_pairs_df = pd.DataFrame()

In [ ]:
def predict_food(
    image_path: str | Path,
    model: nn.Module,
    class_names: list[str],
    top_k: int = 3,
) -> pd.DataFrame:
    """Predict the top food classes for one image."""
    image_path = Path(image_path)
    image = Image.open(image_path).convert("RGB")
    tensor = VAL_TRANSFORMS(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        probabilities = F.softmax(model(tensor), dim=1).squeeze(0)

    top_probabilities, top_indices = torch.topk(probabilities, top_k)
    return pd.DataFrame(
        {
            "rank": range(1, top_k + 1),
            "label": [class_names[idx] for idx in top_indices.cpu().numpy()],
            "probability": top_probabilities.cpu().numpy(),
        }
    )


def collect_error_examples(
    prediction_df: pd.DataFrame,
    max_examples: int = CFG.ERROR_EXAMPLE_COUNT,
) -> pd.DataFrame:
    """Collect high-confidence errors from saved prediction rows."""
    if prediction_df.empty:
        return pd.DataFrame()

    return (
        prediction_df[~prediction_df["is_correct"]]
        .sort_values("confidence", ascending=False)
        .head(max_examples)
        .reset_index(drop=True)
    )


def save_error_grid(error_df: pd.DataFrame) -> None:
    """Save a grid of qualitative error examples."""
    if error_df.empty:
        print("No error examples available.")
        return

    cols = 4
    rows = int(np.ceil(len(error_df) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, error_df.iterrows()):
        image = Image.open(row["path"]).convert("RGB")
        ax.imshow(image)
        ax.set_title(
            f"True: {row['true_label']}\nPred: {row['pred_label']} "
            f"({row['confidence']:.1%})",
            fontsize=9,
        )
        ax.axis("off")

    for ax in axes[len(error_df):]:
        ax.axis("off")

    plt.tight_layout()
    figure_path = CFG.FIGURES_DIR / "qualitative_error_examples.png"
    plt.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()


if final_model is not None:
    error_examples_df = collect_error_examples(test_pred_df)
    error_examples_df.to_csv(CFG.RESULTS_DIR / "qualitative_error_examples.csv", index=False)
    display(error_examples_df)
    save_error_grid(error_examples_df)
else:
    print("Qualitative error analysis skipped because final_model is not loaded.")